In [0]:
pip install Faker

In [0]:
# =============================================================
# NOTEBOOK:  02_bronze_batch_ingestion_transactions
# PURPOSE:   Auto Loader pipeline for ERP transactions data
# SOURCE:    ERP System → PARQUET files
# TARGET:    Bronze Delta Table (bronze/erp/transactions/)
# AUTHOR:    Your Name
# DATE:      Today's date
# PROJECT:   RetailMax Lakehouse
# =============================================================

import json
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime
from faker import Faker
import random
import uuid



# ── Load Config ───────────────────────────────────────────────
config_path = "/Workspace/Repos/retailmax-lakehouse/retailmax-lakehouse/configs/dev/config.json"

with open(config_path, 'r') as f:
    config = json.load(f)

# ── Storage Paths ─────────────────────────────────────────────
storage_account = config['storage']['account_name']
bronze_path     = config['storage']['bronze_path']
scope_name      = config['secret_scope']

# ── Retrieve Secrets ──────────────────────────────────────────
client_id     = dbutils.secrets.get(scope=scope_name, key="sp-client-id")
client_secret = dbutils.secrets.get(scope=scope_name, key="sp-client-secret")
tenant_id     = dbutils.secrets.get(scope=scope_name, key="sp-tenant-id")

# ── Configure Spark for ADLS ──────────────────────────────────
spark.conf.set(
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net",
    "OAuth"
)
spark.conf.set(
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider"
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net",
    client_id
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net",
    client_secret
)
spark.conf.set(
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net",
    f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
)

print("=" * 60)
print("RetailMax - Bronze Batch Ingestion: Transactions")
print("=" * 60)
print(f"✅ Config loaded")
print(f"✅ Secrets retrieved")
print(f"✅ ADLS configured")
print(f"   Bronze Path: {bronze_path}")
print("=" * 60)


In [0]:
source_path = bronze_path + 'raw_ingest/erp/transactions/'
target_path = bronze_path + "tables/erp/transactions/"
checkpoint_path = bronze_path + "_checkpoints/erp/transactions/"
schema_path = bronze_path + "_schema/erp/transactions/"
bad_records_path = bronze_path + "_quarantine/erp/transactions/"


In [0]:
fake = Faker()

PAYMENT_METHODS = ["Credit Card", "Debit Card", "Cash", "Mobile Pay", "Gift Card"]
CHANNELS = ["In-Store", "Online", "Mobile App"]

def generate_transactions(start_order_id, n_orders=300):
    line_items = []
    
    for order_num in range(n_orders):
        order_id = start_order_id + order_num
        order_date = fake.date_time_between(start_date="-90d", end_date="now")
        customer_id = random.randint(1, 2000)
        store_id = random.randint(1, 20)
        payment_method = random.choice(PAYMENT_METHODS)
        channel = random.choice(CHANNELS)
        
        # Each order has 1-5 line items
        n_lines = random.randint(1, 5)
        
        for line_num in range(1, n_lines + 1):
            quantity = random.randint(1, 10)
            unit_price = round(random.uniform(5.0, 200.0), 2)
            
            line_items.append({
                "transaction_id": order_id,
                "line_number": line_num,
                "customer_id": customer_id,
                "product_id": random.randint(1, 1000),
                "store_id": store_id,
                "quantity": quantity,
                "unit_price": unit_price,
                "line_total": round(quantity * unit_price, 2),
                "transaction_date": order_date.date().isoformat(),
                "transaction_ts": order_date.isoformat(),
                "payment_method": payment_method,
                "channel": channel,
                "is_returned": random.random() < 0.03
            })
    
    return line_items

batch1 = generate_transactions(start_order_id=1, n_orders=300)
batch2 = generate_transactions(start_order_id=301, n_orders=300)

print(f"Batch 1: {len(batch1)} line items")
print(f"Batch 2: {len(batch2)} line items")

# Save as Parquet
df_batch1 = spark.createDataFrame(batch1)
df_batch2 = spark.createDataFrame(batch2)

df_batch1.write.mode("overwrite").parquet(source_path + "transactions_batch1/")
df_batch2.write.mode("overwrite").parquet(source_path + "transactions_batch2/")

print("✅ Parquet files saved to ADLS")

In [0]:
transaction_schema = """
    transaction_id BIGINT,
    line_number BIGINT,
    customer_id BIGINT,
    product_id BIGINT,
    store_id BIGINT,
    quantity BIGINT,
    unit_price DOUBLE,
    line_total DOUBLE,
    transaction_date STRING,
    transaction_ts STRING,
    payment_method STRING,
    channel STRING,
    is_returned BOOLEAN
"""

In [0]:
# Reading with Auto Loader
df = spark.readStream\
    .schema(transaction_schema)\
    .format("cloudFiles")\
    .option("cloudFiles.format", "parquet")\
    .option("cloudFiles.schemaLocation", schema_path)\
    .load(source_path)

# Metadata columns
df = (df
    .withColumn("_source_file", F.input_file_name())
    .withColumn("_load_timestamp", F.current_timestamp())
    .withColumn("_load_date", F.current_date())
    .withColumn("_batch_id", F.lit(uuid.uuid4().hex))
)

# Writing stream
df.writeStream\
    .format("delta")\
    .option("checkpointLocation", checkpoint_path)\
    .trigger(availableNow=True)\
    .partitionBy("_load_date")\
    .start(target_path)\
    .awaitTermination()

In [0]:
df_check = spark.read.parquet(source_path + "transactions_batch1/")
df_check.printSchema()

In [0]:
df = spark.read.format("delta").load(target_path)
print(df.count())
df.show(5)
df.printSchema()